# 1. Métodos del curso y validación

**Proyecto Integrador · Métodos Numéricos · Segunda entrega**
David Castro, Alejandro Pardo, Tomás Rincón · Docente: Edwin Fernando Muñoz
Artículo base: Cartis & Roberts (2019), *A derivative-free Gauss–Newton method*.

Implementa los métodos de álgebra lineal de la Unidad 3, codifica los problemas de prueba del artículo y comprueba que todo sea correcto.

El cuaderno es autónomo: basta con ejecutar todas las celdas en Colab o Jupyter.

In [1]:
import time, csv
from dataclasses import dataclass
from pathlib import Path
from typing import Callable
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares  # solo validación y f* de referencia

## 1. Eliminación de Gauss con pivoteo (Semana 6)

Sigue el notebook de la Semana 6: $m=a_{ji}/a_{ii}$ y $a_{jk}:=a_{jk}-m\,a_{ik}$. Agregamos dos cosas:

- **Pivoteo parcial** (microcurrículo, S11): evita dividir por pivotes nulos o muy pequeños.
- **Varios lados derechos:** DFO-GN resuelve $m$ sistemas con la misma matriz, así que basta una eliminación, igual que la ventaja de LU del Quiz 3.

In [2]:
def eliminacion_gauss(A, B):
    """Eliminación hacia adelante con pivoteo parcial.

    A es n x n; B es n x p (p lados derechos). Con una sola eliminación se
    resuelven los p sistemas, que es la misma ventaja que tiene LU (Quiz 3, 2g).
    El multiplicador es m = a_ji / a_ii y la actualización a_jk := a_jk - m a_ik.

    Devuelve U, B modificada y razon_pivotes = min|u_ii| / max|u_ii|. Si la razón
    es casi cero, A está cerca de ser singular.
    """
    U = np.array(A, dtype=float)
    C = np.array(B, dtype=float)
    n = U.shape[0]
    for i in range(n - 1):
        # Pivoteo parcial: se sube la fila con el mayor |a_ji| para que |m| <= 1
        # y los errores de redondeo no se amplifiquen.
        p = i + int(np.argmax(np.abs(U[i:, i])))
        if U[p, i] == 0.0:
            raise np.linalg.LinAlgError("Matriz singular en la eliminación de Gauss.")
        if p != i:
            U[[i, p]] = U[[p, i]]
            C[[i, p]] = C[[p, i]]
        for j in range(i + 1, n):
            m = U[j, i] / U[i, i]
            U[j, i] = 0.0
            U[j, i + 1:] -= m * U[i, i + 1:]
            C[j] -= m * C[i]
    pivotes = np.abs(np.diag(U))
    razon = pivotes.min() / pivotes.max() if pivotes.max() > 0 else 0.0
    return U, C, razon

In [3]:
def sustitucion_atras(U, c):
    """Resuelve Ux = c con U triangular superior. c puede tener varias columnas."""
    n = U.shape[0]
    x = np.array(c, dtype=float)
    for i in range(n - 1, -1, -1):
        if U[i, i] == 0.0:
            raise np.linalg.LinAlgError("Pivote nulo en la sustitución hacia atrás.")
        x[i] = (x[i] - U[i, i + 1:] @ x[i + 1:]) / U[i, i]
    return x

In [4]:
def resolver_gauss(A, B):
    """Resuelve AX = B. Devuelve X y la razón de pivotes."""
    U, C, razon = eliminacion_gauss(A, B)
    return sustitucion_atras(U, C), razon

## 2. QR y mínimos cuadrados (Semana 6, Sesión 12)

Gram-Schmidt modificado: $v_j := v_j - r_{ij}q_i$. El problema $\min\|Ax-b\|_2$ se resuelve con $Rx=Q^Tb$.

In [5]:
def gram_schmidt_modificado(A, tol=1e-13):
    """QR reducida A = QR (m >= n), igual al notebook de la Semana 6.

    Cada columna v_j se actualiza justo después de cada proyección:
    v_j := v_j - r_ij q_i. tol es relativa a la mayor columna de A.
    """
    V = np.array(A, dtype=float)
    m, n = V.shape
    if m < n:
        raise ValueError("Se requiere m >= n.")
    Q = np.zeros((m, n))
    R = np.zeros((n, n))
    escala = max(np.linalg.norm(V, axis=0).max(), 1e-300)
    for i in range(n):
        R[i, i] = np.linalg.norm(V[:, i])
        if R[i, i] <= tol * escala:
            raise np.linalg.LinAlgError("Columnas linealmente dependientes.")
        Q[:, i] = V[:, i] / R[i, i]
        for j in range(i + 1, n):
            R[i, j] = Q[:, i] @ V[:, j]
            V[:, j] -= R[i, j] * Q[:, i]
    return Q, R

In [6]:
def minimos_cuadrados_qr(A, b):
    """Solución de min ||Ax - b||_2 mediante Rx = Q^T b (Sesión 12).

    Q^T b se calcula proyectando b de forma secuencial, con la misma idea de
    Gram-Schmidt modificado; así se conserva mejor la precisión si Q pierde
    algo de ortogonalidad.
    """
    Q, R = gram_schmidt_modificado(A)
    v = np.array(b, dtype=float)
    c = np.zeros(R.shape[0])
    for i in range(R.shape[0]):
        c[i] = Q[:, i] @ v
        v -= c[i] * Q[:, i]
    return sustitucion_atras(R, c)

## 3. Problemas de prueba (Tabla 2 del artículo)

Definen $f(x)=\tfrac12\|r(x)\|^2$. Son funciones estándar para evaluar optimizadores. Elegimos siete con $2\le n\le 6$ y características distintas (residuo cero y no nulo, $m=n$ y $m>n$, $J$ singular en la solución).

`Evaluador` cuenta las evaluaciones de $r$ y genera el ruido de la sección 5.2 del artículo.

In [7]:
@dataclass
class Problema:
    numero: int            # número en la Tabla 2
    nombre: str
    r: Callable            # residuos, R^n -> R^m
    J: Callable            # Jacobiano analítico, m x n
    x0: np.ndarray
    dos_f0_tabla: float
    dos_f_estrella: float

    @property
    def f_estrella(self):
        return self.dos_f_estrella / 2

    @property
    def n(self):
        return len(self.x0)

    @property
    def m(self):
        return len(self.r(self.x0))


def _rosenbrock_r(x):
    return np.array([10 * (x[1] - x[0] ** 2), 1 - x[0]])


def _rosenbrock_J(x):
    return np.array([[-20 * x[0], 10.0], [-1.0, 0.0]])


def _freudenstein_r(x):
    return np.array([-13 + x[0] + ((5 - x[1]) * x[1] - 2) * x[1],
                     -29 + x[0] + ((x[1] + 1) * x[1] - 14) * x[1]])


def _freudenstein_J(x):
    return np.array([[1.0, 10 * x[1] - 3 * x[1] ** 2 - 2],
                     [1.0, 3 * x[1] ** 2 + 2 * x[1] - 14]])


def _theta(x):
    t = np.arctan(x[1] / x[0]) / (2 * np.pi) if x[0] != 0 else 0.25
    return t + 0.5 if x[0] < 0 else t


def _helical_r(x):
    return np.array([10 * (x[2] - 10 * _theta(x)),
                     10 * (np.hypot(x[0], x[1]) - 1), x[2]])


def _helical_J(x):
    d2 = x[0] ** 2 + x[1] ** 2
    d = np.sqrt(d2)
    return np.array([[100 * x[1] / (2 * np.pi * d2), -100 * x[0] / (2 * np.pi * d2), 10.0],
                     [10 * x[0] / d, 10 * x[1] / d, 0.0],
                     [0.0, 0.0, 1.0]])


_BARD_Y = np.array([0.14, 0.18, 0.22, 0.25, 0.29, 0.32, 0.35, 0.39, 0.37, 0.58,
                    0.73, 0.96, 1.34, 2.10, 4.39])
_BARD_U = np.arange(1.0, 16.0)
_BARD_V = 16.0 - _BARD_U
_BARD_W = np.minimum(_BARD_U, _BARD_V)


def _bard_r(x):
    return _BARD_Y - (x[0] + _BARD_U / (_BARD_V * x[1] + _BARD_W * x[2]))


def _bard_J(x):
    d = (_BARD_V * x[1] + _BARD_W * x[2]) ** 2
    return np.column_stack((-np.ones(15), _BARD_U * _BARD_V / d, _BARD_U * _BARD_W / d))


def _powell_r(x):
    return np.array([x[0] + 10 * x[1], np.sqrt(5) * (x[2] - x[3]),
                     (x[1] - 2 * x[2]) ** 2, np.sqrt(10) * (x[0] - x[3]) ** 2])


def _powell_J(x):
    a = 2 * (x[1] - 2 * x[2])
    b = 2 * np.sqrt(10) * (x[0] - x[3])
    s = np.sqrt(5)
    return np.array([[1.0, 10.0, 0.0, 0.0], [0.0, 0.0, s, -s],
                     [0.0, a, -2 * a, 0.0], [b, 0.0, 0.0, -b]])


_KO_Y = np.array([0.1957, 0.1947, 0.1735, 0.1600, 0.0844, 0.0627, 0.0456,
                  0.0342, 0.0323, 0.0235, 0.0246])
_KO_U = np.array([4.0, 2.0, 1.0, 0.5, 0.25, 0.1670, 0.1250, 0.1000, 0.0833,
                  0.0714, 0.0625])


def _kowalik_r(x):
    u = _KO_U
    return _KO_Y - x[0] * (u ** 2 + u * x[1]) / (u ** 2 + u * x[2] + x[3])


def _kowalik_J(x):
    u = _KO_U
    num = u ** 2 + u * x[1]
    den = u ** 2 + u * x[2] + x[3]
    return np.column_stack((-num / den, -x[0] * u / den,
                            x[0] * num * u / den ** 2, x[0] * num / den ** 2))


def _watson_r(x):
    n = len(x)
    t = np.arange(1, 30) / 29.0
    pot = np.vander(t, n, increasing=True)            # t^(j-1), j = 1..n
    suma = pot @ x
    deriv = pot[:, :-1] @ (np.arange(1, n) * x[1:])   # suma de (j-1) x_j t^(j-2)
    return np.concatenate((deriv - suma ** 2 - 1, [x[0], x[1] - x[0] ** 2 - 1]))


def _watson_J(x):
    n = len(x)
    t = np.arange(1, 30) / 29.0
    pot = np.vander(t, n, increasing=True)
    suma = pot @ x
    J = np.zeros((31, n))
    J[:29, 0] = -2 * suma
    J[:29, 1:] = pot[:, :-1] * np.arange(1, n) - 2 * suma[:, None] * pot[:, 1:]
    J[29, 0] = 1.0
    J[30, 0] = -2 * x[0]
    J[30, 1] = 1.0
    return J


PROBLEMAS = [
    Problema(7, "Rosenbrock", _rosenbrock_r, _rosenbrock_J, np.array([-1.2, 1.0]), 24.2, 0.0),
    Problema(13, "Freudenstein & Roth", _freudenstein_r, _freudenstein_J,
             np.array([0.5, -2.0]), 400.5, 48.98425),
    Problema(9, "Helical Valley", _helical_r, _helical_J, np.array([-1.0, 0.0, 0.0]), 2500.0, 0.0),
    Problema(15, "Bard", _bard_r, _bard_J, np.array([1.0, 1.0, 1.0]), 41.68170, 8.214877e-3),
    Problema(11, "Powell Singular", _powell_r, _powell_J,
             np.array([3.0, -1.0, 0.0, 1.0]), 215.0, 0.0),
    Problema(17, "Kowalik & Osborne", _kowalik_r, _kowalik_J,
             np.array([0.25, 0.39, 0.415, 0.39]), 5.313172e-3, 3.075056e-4),
    Problema(19, "Watson", _watson_r, _watson_J, 0.5 * np.ones(6), 16.43083, 2.287670e-3),
]


class Evaluador:
    """Envuelve r(x): cuenta evaluaciones y, si se pide, agrega ruido (sección 5.2 del artículo).

    ruido = None, 'multiplicativo' (r_i (1 + e)) o 'aditivo' (r_i + e), con
    e ~ N(0, sigma^2) independiente por componente y por evaluación.
    """

    def __init__(self, problema, ruido=None, sigma=1e-2, semilla=None):
        self.problema = problema
        self.ruido = ruido
        self.sigma = sigma
        self.rng = np.random.default_rng(semilla)
        self.n_eval = 0
        self.n_eval_J = 0

    def r(self, x):
        self.n_eval += 1
        valor = self.problema.r(x)
        if self.ruido is None:
            return valor
        e = self.rng.normal(0.0, self.sigma, valor.shape)
        return valor * (1 + e) if self.ruido == "multiplicativo" else valor + e

    def J(self, x):
        self.n_eval_J += 1
        return self.problema.J(x)

    def f_real(self, x):
        """f sin ruido y sin contar: solo para medir, nunca la usa el algoritmo."""
        return 0.5 * float(np.sum(self.problema.r(x) ** 2))

## 4. Validación

Aquí sí se usan NumPy y SciPy de alto nivel, porque la guía las permite para validar (sección 1, regla 4).

- Nuestras rutinas contra `np.linalg`, incluidos un ejemplo de la Semana 6 y el ejercicio 3 del Quiz 3.
- $2f(x_0)$ de cada problema contra la Tabla 2 del artículo.
- $J$ analítico contra diferencias centradas ($O(h^2)$, Unidad 2).
- $2f^*$ de la Tabla 2 contra `least_squares`.

In [8]:
def validar_algebra():
    rng = np.random.default_rng(2026)
    # Ejemplo del notebook de la Semana 6, con solución calculable a mano.
    A = np.array([[2.0, 1, -3], [4, 3, -5], [-2, 2, 4]])
    b = np.array([5.0, 9, 7])
    x, _ = resolver_gauss(A, b)
    assert np.allclose(x, np.linalg.solve(A, b))
    # Sin pivoteo el primer pivote sería cero y el algoritmo se dividiría por 0.
    x, _ = resolver_gauss(np.array([[0.0, 1], [1, 1]]), np.array([1.0, 2]))
    assert np.allclose(x, [1, 1])
    # Quiz 3, ejercicio 3: mínimos cuadrados con QR.
    A = np.array([[1.0, 0], [1, 1], [1, 2]])
    b = np.array([1.0, 2, 4])
    assert np.allclose(minimos_cuadrados_qr(A, b), np.linalg.lstsq(A, b, rcond=None)[0])
    errores = []
    for _ in range(200):
        n = rng.integers(2, 8)
        m = n + rng.integers(0, 30)
        A = rng.normal(size=(m, n))
        b = rng.normal(size=m)
        Q, R = gram_schmidt_modificado(A)
        errores.append(max(np.abs(Q @ R - A).max(), np.abs(Q.T @ Q - np.eye(n)).max()))
        assert np.allclose(minimos_cuadrados_qr(A, b), np.linalg.lstsq(A, b, rcond=None)[0])
        C = rng.normal(size=(n, n))
        d = rng.normal(size=(n, 3))
        assert np.allclose(resolver_gauss(C, d)[0], np.linalg.solve(C, d))
    print(f"Álgebra correcta en 200 casos. Máx. error de QR y de ortogonalidad: {max(errores):.1e}")


validar_algebra()

Álgebra correcta en 200 casos. Máx. error de QR y de ortogonalidad: 8.0e-16


In [9]:
def validar_problemas():
    rng = np.random.default_rng(2026)
    print(f"{'Problema':24}{'2f(x0)':>14}{'Tabla 2':>14}{'error J':>11}{'2f SciPy':>14}{'2f* Tabla':>14}")
    for p in PROBLEMAS:
        dos_f0 = float(np.sum(p.r(p.x0) ** 2))
        assert abs(dos_f0 - p.dos_f0_tabla) <= 1e-5 * p.dos_f0_tabla, p.nombre
        # J analítico contra diferencias centradas, O(h^2), en un punto genérico.
        x = p.x0 + 0.01 * rng.normal(size=p.n) + 0.01
        h = 1e-6
        Jdf = np.column_stack([(p.r(x + h * e) - p.r(x - h * e)) / (2 * h) for e in np.eye(p.n)])
        error = np.abs(Jdf - p.J(x)).max() / max(np.abs(Jdf).max(), 1)
        assert error < 1e-6, p.nombre
        ref = least_squares(p.r, p.x0, jac=p.J, xtol=1e-14, ftol=1e-14, gtol=1e-14)
        print(f"{p.nombre:24}{dos_f0:14.6g}{p.dos_f0_tabla:14.6g}{error:11.1e}"
              f"{2 * ref.cost:14.6e}{p.dos_f_estrella:14.6e}")
    print("Los problemas coinciden con la Tabla 2 y los Jacobianos con diferencias finitas.")


validar_problemas()

Problema                        2f(x0)       Tabla 2    error J      2f SciPy     2f* Tabla
Rosenbrock                        24.2          24.2    7.5e-11  0.000000e+00  0.000000e+00
Freudenstein & Roth              400.5         400.5    6.5e-11  4.898425e+01  4.898425e+01
Helical Valley                    2500          2500    2.0e-10  3.146523e-59  0.000000e+00
Bard                           41.6817       41.6817    2.0e-10  8.214877e-03  8.214877e-03
Powell Singular                    215           215    1.2e-10  3.409308e-20  0.000000e+00
Kowalik & Osborne           0.00531317    0.00531317    4.4e-11  3.075056e-04  3.075056e-04
Watson                         16.4308       16.4308    2.4e-10  2.287670e-03  2.287670e-03
Los problemas coinciden con la Tabla 2 y los Jacobianos con diferencias finitas.


**Lectura:** si las celdas no fallan, las rutinas coinciden con las de referencia y los problemas quedaron bien codificados. Ninguna función de alto nivel se usa dentro de los algoritmos.